In [1]:
#GV 13.02.2025
#Multiclass labels
#It collects with multilabels considering onsets and converts to single label with singlelabel
#Then it trains a multilabel classifier. The results are not so interesting, 
#maybe because the label collection is not optimal

import sys
sys.path.extend(['src', '../src']) 
# Import all ML orchestration functions
from amo.ml_orchestration import (
    ml_exp_with_timing,  # Main experiment function with timing preservation
    extract_timing_structure,
    save_midi_with_exact_timing_structure,
    KerasClassifierWrapper,
    build_lstm_classifier,
    build_transformer_classifier,
    clf_predict,
    defineXy,           # Data preparation function
    split_and_encode    # Train/test split with encoding
)
# Import MIDI processing functions  
from amo.midi2df2midi import midi_to_dataframe, save_midi_from_df
from amo.mappings import fill_quaterna_columns, learn_quaterna_mapping
# Import amo funtion
from amo.ml_orchestration import amo_with_doublings_multiclass

import pandas as pd
import numpy as np
from collections import defaultdict


In [2]:
import os

In [3]:
from xgboost import XGBClassifier

In [4]:
def defineXy_all_onsets(nmat, ytarget="track-channel"):
    """Extract features and labels from note matrix."""
    if ytarget == "program":
        X = nmat[:, 4:8]  # onset, duration, pitch, velocity
        y = nmat[:, 3]    # program
    elif ytarget == "instrument-name":
        X = nmat[:, 4:8]  # onset, duration, pitch, velocity (indices 4-7)
        y = nmat[:, 1].astype(str)
    else:
        # X: onset, duration, pitch, velocity, y=track_channel
        X = nmat[:, 4:8]  # onset, duration, pitch, velocity
        A = nmat[:, 0].astype(str)
        B = nmat[:, 2].astype(str)
        y = np.char.add(np.char.add(A, '_'), B)  # Label is "track_channel"
    return X, y

In [5]:
def clf_predict_back(X, y, le, mapping, ytarget):
    #"""Predict orchestration using trained model."""
    #y_pred = model.predict(X2)
    #print("Predictions ", np.unique(y_pred))
    # inverse transform to obtained the original labels:
    y_pred_orig = le.inverse_transform(y) #track_channel
    # Fill columns track, track name, channel, program using the mapping
    #fill_quaterna_columns(y, map_array, ytarget="track-channel"):
    new_cols = fill_quaterna_columns(y_pred_orig, mapping, ytarget)
    print("Predictions map", np.unique(y_pred_orig))
    nmat = np.concatenate((new_cols, X), axis=1)
    return nmat

In [6]:
def clf_predict_multi(X2, le, model, mapping, ytarget):
    """Predict orchestration using trained model."""
    y_pred = model.predict(X2)
    print("Predictions ", np.unique(y_pred))
    # inverse transform to obtained the original labels:
    y_pred_orig = le.inverse_transform(y_pred) #track_channel
    # Fill columns track, track name, channel, program using the mapping
    new_cols = fill_quaterna_columns(y_pred_orig, mapping, ytarget)
    print("Predictions map", np.unique(y_pred_orig))
    nmat = np.concatenate((new_cols, X2), axis=1)
    return nmat

In [7]:
#X2 (n,4) y2 array([6, 7, 6, ..., 7, 6, 7])
def Xy_to_midi(X, y, mapping, le, filename, fileref, ytarget="track-channel"):   
    #X, _, y, _, le = split_and_encode(X, y, test_size=0)
    X=clf_predict_back(X, y, le, mapping, ytarget)
    # Convert to DataFrame
    # Get original ticks_per_beat for precise timing
    
    dfdata = pd.DataFrame(X, columns=[
        'track number', 'track name', 'channel', 'program',
        'onset in quarter notes', 'duration in quarter notes', 'pitch', 'velocity'
    ])
    
    # Fix data types
    dfdata['track number'] = dfdata['track number'].astype(int)
    dfdata['channel'] = dfdata['channel'].astype(int)
    dfdata['program'] = dfdata['program'].astype(int)
    dfdata['pitch'] = dfdata['pitch'].astype(int)
    dfdata['velocity'] = dfdata['velocity'].astype(int)
    dfdata['onset in quarter notes'] = dfdata['onset in quarter notes'].astype(float)
    dfdata['duration in quarter notes'] = dfdata['duration in quarter notes'].astype(float)
    dfdata['track name'] = dfdata['track name'].astype(str)
    save_midi_with_exact_timing_structure(dfdata, filename, reference_midi_path=fileref)
    #save_midi_from_df(dfdata, output_path, ticks_per_beat=480)

In [8]:
#12.2.2026
'''
Create a python function y_multi=multilabels(X,y) such that y_multi is the one hot-encoding of y.
create another python function X2,y2=singlelabel(X,y_multi) that returns y. See for example:
X
array([[0, 0.1, 66, 112],
       [1, 0.3, 47, 40],
       [0, 0.125, 62, 112],
       [2.5, 0.125, 47, 105],
       [1, 0.125, 47, 107]], dtype=object)
y
array([0, 1, 2, 3, 1])
y_multi
array([[1, 0, 1, 0],
       [0, 1, 0, 0],
       [1, 0, 1, 0],
       [0, 0, 0, 1],
       [0, 1, 0, 0]])
X2
array([[0, 0.1, 66, 112],
       [0, 0.1, 66, 112],
       [1, 0.3, 47, 40],
       [0, 0.125, 62, 112],
       [0, 0.125, 62, 112],
       [2.5, 0.125, 47, 105],
       [1, 0.125, 47, 107]], dtype=object)
y2
array([0, 2, 1, 0, 2, 3, 1])

'''
#import numpy as np

def multilabels(X, y, onset_round=10):
    """
    Create a multi-label (multi-hot) encoding of y per *onset*.

    For each onset value in X[:,0], we collect all labels y that appear at that onset.
    Then every row having that onset gets the same multi-hot vector.

    Assumptions:
      - y is integer-coded labels (e.g., 0..K-1). If not, encode first.

    Parameters
    ----------
    X : (n, m) array-like. Uses column 0 as onset.
    y : (n,) array-like of int labels.
    onset_round : int. Round onsets to this many decimals before grouping (helps floats).

    Returns
    -------
    y_multi : (n, K) int array of 0/1, where K = max(y)+1
    """
    X = np.asarray(X)
    y = np.asarray(y)

    if X.ndim != 2 or X.shape[0] != y.shape[0]:
        raise ValueError("X must be 2D and y must be 1D with matching length.")
    if not np.issubdtype(y.dtype, np.integer):
        raise ValueError("y must be integer-coded (e.g., 0..K-1). Encode labels before calling.")

    K = int(y.max()) + 1
    n = len(y)

    # group by (rounded) onset
    onset_key = np.round(X[:, 0].astype(float), onset_round)

    # build onset -> set(labels)
    onset_to_labels = {}
    for ok, yi in zip(onset_key, y):
        onset_to_labels.setdefault(ok, set()).add(int(yi))

    # fill y_multi
    y_multi = np.zeros((n, K), dtype=int)
    for i, ok in enumerate(onset_key):
        labs = onset_to_labels[ok]
        y_multi[i, list(labs)] = 1

    return y_multi


def singlelabel(X, y_multi, classes=None):
    """
    Convert a multi-hot matrix back into single-label rows by expanding X.

    Each (row i, class j) where y_multi[i,j]==1 becomes one output row:
      X2 contains X[i] repeated
      y2 contains j (or classes[j] if classes is provided)

    Parameters
    ----------
    X : (n, m) array-like
    y_multi : (n, K) array-like of 0/1
    classes : optional array-like of original label values of length K.
              If None, returns integer labels 0..K-1.

    Returns
    -------
    X2 : (nnz, m) array
    y2 : (nnz,) array
    """
    X = np.asarray(X)
    y_multi = np.asarray(y_multi)

    if X.ndim != 2 or y_multi.ndim != 2 or X.shape[0] != y_multi.shape[0]:
        raise ValueError("X must be 2D and y_multi must be 2D with matching number of rows.")

    rows, cols = np.where(y_multi == 1)

    X2 = X[rows]
    if classes is None:
        y2 = cols
    else:
        classes = np.asarray(classes)
        if len(classes) != y_multi.shape[1]:
            raise ValueError("classes must have length equal to number of columns in y_multi.")
        y2 = classes[cols]

    return X2, y2

In [9]:
#get the parent directory
cwd = os.getcwd()
parent = os.path.normpath(os.path.join(cwd, '..', '..'))
path_AMO = parent + '/AutomaticMusicOrchestration'
model_path = path_AMO + '/'
#print(parent)#.../test
#print(path_AMO)

In [10]:
filein=parent+'/music/LOP_database_06_09_17/hand_picked_Spotify/39/swan_lake_09_orch.mid'

In [11]:
fileout = path_AMO + '/data/samples/midis/fur-elise.mid'

In [12]:
# Load and process source file
dfnmat = midi_to_dataframe(filein)
dfnmat = dfnmat.sort_values(
        ['onset in quarter notes','duration in quarter notes', 'track number'],
        ascending=[True, True, True])
nmat = dfnmat.to_numpy()  
mapping = learn_quaterna_mapping(nmat, ytarget="track-channel")
#print("mapping", mapping)
    
#X, y = defineXy_all_onsets(nmat, ytarget="track-channel")

In [13]:
# Load and process source file
dfnmat2 = midi_to_dataframe(fileout)
dfnmat2 = dfnmat2.sort_values(
        ['onset in quarter notes','duration in quarter notes', 'track number'],
        ascending=[True, True, True])
nmat2 = dfnmat2.to_numpy() 

In [14]:
X,y=defineXy_all_onsets(nmat) #X array (n samples,4), y array(['17_11', '16_0', '18_12', ..., '13_10', '13_10', '13_10'],dtype='<U5')

In [15]:
X, _, y, _, le = split_and_encode(X, y, test_size=0) #X array (n samples,4), y array([7, 6, 8, ..., 3, 3, 3])

y_train, test size: 0 , labels: ['10_7' '11_7' '12_8' '13_10' '14_9' '15_10' '16_0' '17_11' '18_12'
 '19_13' '1_1' '20_14' '21_15' '2_1' '3_2' '4_3' '5_4' '6_5' '7_5' '8_6'
 '9_6']


In [16]:
y_multi = multilabels(X, y)# y_multi is multi-hot encoding

clf = XGBClassifier(tree_method="hist")
clf.fit(X, y_multi)
#y_pred=clf_predict(nmat2[:, 4:8], le, clf, mapping)
y_pred=clf.predict(nmat2[:, 4:8]) # y_pred is multi-hot encoding

In [17]:
X2, y2 = singlelabel(nmat2[:, 4:8], y_pred) #X2 (n,4) y2 array([6, 7, 6, ..., 7, 6, 7])

In [18]:
#Xy_to_midi(X2, y2, mapping, 'test.mid', ytarget="track-channel")
filetest = path_AMO + '/data/samples/midis/test3.mid'
Xy_to_midi(X2, y2, mapping, le, filetest, fileout, ytarget="track-channel")

Predictions map ['12_8' '16_0' '17_11' '19_13' '1_1' '2_1' '4_3' '9_6']

=== SAVING WITH EXACT TIMING STRUCTURE ===
ticks_per_beat: 480
Found 11 timing events:
  1. time_signature: 1/8 at 0.000 quarters (0 ticks)
  2. key_signature: Key: C at 0.000 quarters (0 ticks)
  3. tempo: 72.0 BPM at 0.000 quarters (0 ticks)
  4. key_signature: Key: C at 0.000 quarters (0 ticks)
  5. time_signature: 3/8 at 0.500 quarters (240 ticks)
  6. time_signature: 2/8 at 11.000 quarters (5280 ticks)
  7. time_signature: 1/8 at 12.000 quarters (5760 ticks)
  8. tempo: 72.0 BPM at 12.000 quarters (5760 ticks)
  9. time_signature: 3/8 at 12.500 quarters (6000 ticks)
  10. time_signature: 3/8 at 23.000 quarters (11040 ticks)
  11. time_signature: 2/8 at 186.500 quarters (89520 ticks)
Using ticks_per_beat: 480
✓ Added 1/8 at 0.000 quarters
✓ Added key signature Am at 0.000 quarters
✓ Added tempo 72.0 BPM at 0.000 quarters
⚠️ Skipped duplicate key signature Am at 0.000 quarters
✓ Added 3/8 at 0.500 quarters
✓ Ad